# TFM DSMarket — Modelización

Notebook de modelización para la predicción de ventas a 28 días (Tarea 3).

**Input:** `df_preprocessed.feather` generado por `preprocesamiento.ipynb`  
**Output:** predicciones de ventas a nivel producto x tienda para los próximos 28 días

---

## Tabla de contenidos

1. Carga y exploración rápida
2. Separación temporal train/test
3. Feature engineering
4. Modelo 1 - XGBoost
5. Modelo 2 - Prophet
6. Modelo 3 - SARIMAX
7. Comparativa de modelos y selección
8. Generación de predicciones finales

## 0. Imports y configuración

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error


from prophet import Prophet

from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/data_dsmarket/'
except ImportError:
    DATA_PATH = 'data_dsmarket/'

RANDOM_SEED = 42

---
## 1. Carga y exploración rápida

In [2]:
df = pd.read_feather(DATA_PATH + 'df_preprocessed.feather')
df['date'] = pd.to_datetime(df['date'])

# Reducimos a 3 tiendas para desarrollo en local, me salían 46 millones de filas 
stores_sample = ['NYC_1', 'BOS_1', 'PHI_1']
df = df[df['store_code'].isin(stores_sample)].reset_index(drop=True)

print(f'Shape reducido: {df.shape}')
print(f'Series unicas: {df["id"].nunique()}')

Shape reducido: (13890873, 22)
Series unicas: 9147


In [3]:
df.dtypes

id                                    category
item                                  category
category                              category
department                              object
store                                 category
store_code                            category
region                                category
d                                       object
sales                                    int64
date                            datetime64[us]
weekday                                 object
event                                   object
yearweek                                object
sell_price                             float64
season                                  object
pay_period                              object
ingresos                               float64
is_holiday                               int64
year                                     int64
week                                     int64
month                                    int32
indice_estaci

---
## 2. Separación temporal train/test

En series temporales **no se puede hacer un split aleatorio**. Si mezclas fechas,
el modelo aprende del futuro para predecir el pasado, lo que infla artificialmente
las métricas pero falla en producción.

La regla es: todo lo anterior a la fecha de corte es train, todo lo posterior es test.
Reservamos los últimos **28 días** como test, que es el horizonte de predicción del enunciado.

In [4]:
fecha_corte = df['date'].max() - pd.Timedelta(days=28)
print(f'Fecha de corte: {fecha_corte}')

Fecha de corte: 2016-03-27 00:00:00


---
## 3. Feature engineering

El feature engineering es el puente entre los patrones observados en el EDA
y lo que el modelo puede aprender. Construimos tres bloques de features.

**Regla fundamental:** cada feature calculada para el dia `t`
solo puede usar información de dias anteriores a `t`. Si usamos datos futuros
el modelo aprende trampeando (data leakage) y fallará en producción.

**Features que vienen del preprocesamiento** (ya están en el df):
`sell_price`, `is_holiday`, `season`, `pay_period`, `indice_estacional_store_item`

**Features que construimos aquí:**
- Variables de calendario extraídas de la fecha
- Lag features: ventas de hace N dias
- Rolling window features: estadísticos sobre ventanas de dias pasados

### 3.1 Variables de calendario

Extraemos información de la fecha que el modelo no puede inferir por sí solo.
Son seguras frente al data leakage porque no dependen de las ventas:
siempre sabemos qué dia de la semana o qué mes será cualquier fecha futura.

In [5]:
def add_calendar_features(df):
    df = df.copy()
    df['dayofweek']  = df['date'].dt.dayofweek
    df['dayofmonth'] = df['date'].dt.day
    df['dayofyear']  = df['date'].dt.dayofyear
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
    df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)
    return df

df = add_calendar_features(df)
print('Variables de calendario añadidas.')

Variables de calendario añadidas.


### 3.2 Lag features

Un lag feature son las ventas de hace N dias para ese mismo producto x tienda.
Es la forma de darle al modelo memoria sobre el comportamiento pasado de cada serie.

**Por qué los lags tienen que ser >= 28:**
Si predecimos hasta t+28, el lag mínimo utilizable es 28 dias.
Un lag de 7 para predecir t+1 estaría bien, pero para predecir t+28
necesitaríamos las ventas de t+21, que aún no conocemos.
Usamos lags de 28, 35 y 42 para cubrir el horizonte completo con seguridad.

In [6]:
def add_lag_features(df, lags=[28, 35, 42]):
    df = df.sort_values(['id', 'date']).copy()
    for lag in lags:
        df[f'lag_{lag}'] = df.groupby('id')['sales'].shift(lag)
    return df

df = add_lag_features(df)
lag_cols = [c for c in df.columns if c.startswith('lag_')]
print(f'Lag features creadas: {lag_cols}')

Lag features creadas: ['lag_28', 'lag_35', 'lag_42']


### 3.3 Rolling window features

Calculamos estadísticos (media, std, mediana) sobre ventanas de dias pasados.
Capturan el nivel reciente de ventas y su variabilidad.

**Por qué usamos shift(28) antes del rolling:**
Al predecir el horizonte completo t+1 a t+28, la ventana solo puede
mirar hasta t. Sin el shift, cuando calculamos la rolling del dia t+14
incluiríamos ventas de t+1 a t+13 que aún no tenemos disponibles.
El shift(28) garantiza que solo usamos ventas que tenemos en el peor caso.

In [7]:
def add_rolling_features(df, windows=[7, 14, 28], shift=28):
    df = df.sort_values(['id', 'date']).copy()
    for window in windows:
        shifted = df.groupby('id')['sales'].shift(shift)
        df[f'rolling_mean_{window}']   = shifted.groupby(df['id']).transform(
            lambda x: x.rolling(window, min_periods=1).mean())
        df[f'rolling_std_{window}']    = shifted.groupby(df['id']).transform(
            lambda x: x.rolling(window, min_periods=1).std())
        df[f'rolling_median_{window}'] = shifted.groupby(df['id']).transform(
            lambda x: x.rolling(window, min_periods=1).median())
    return df

df = add_rolling_features(df)
rolling_cols = [c for c in df.columns if c.startswith('rolling_')]
print(f'Rolling features creadas: {rolling_cols}')

Rolling features creadas: ['rolling_mean_7', 'rolling_std_7', 'rolling_median_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_median_14', 'rolling_mean_28', 'rolling_std_28', 'rolling_median_28']


In [8]:
df_train = df[df['date'] <= fecha_corte].copy()
df_test  = df[df['date'] >  fecha_corte].copy()

df_train = df_train.dropna(subset=lag_cols).reset_index(drop=True)

print(f'Train: {df_train["date"].min()} a {df_train["date"].max()} ({df_train["date"].nunique()} dias)')
print(f'Test:  {df_test["date"].min()} a {df_test["date"].max()} ({df_test["date"].nunique()} dias)')
print(f'NaN en lags del test: {df_test[lag_cols].isna().sum().sum()}')

Train: 2011-03-12 00:00:00 a 2016-03-27 00:00:00 (1843 dias)
Test:  2016-03-28 00:00:00 a 2016-04-24 00:00:00 (28 dias)
NaN en lags del test: 0


### 3.4 Integración del cluster (pendiente)

> El notebook de clustering está en desarrollo. Una vez finalizado,
> se añadirá aquí la columna `cluster_id` como feature adicional.
> El resto del pipeline no requiere ninguna modificación.

```python
# Cuando esté disponible:
# df_clusters = pd.read_csv(DATA_PATH + 'clusters.csv')  # columnas: item, cluster_id
# df_train = df_train.merge(df_clusters, on='item', how='left')
# df_test  = df_test.merge(df_clusters, on='item', how='left')
```

### 3.5 Resumen de features disponibles para el modelo

In [9]:
# Features categóricas
CAT_FEATURES = ['category', 'department', 'store_code', 'region', 'season', 'pay_period', 'weekday']

# Features numéricas
NUM_FEATURES = (
    ['sell_price', 'is_holiday', 'is_weekend',
     'dayofweek', 'dayofmonth', 'weekofyear', 'month', 'year', 'week',
     'indice_estacional_store_item']
    + lag_cols
    + rolling_cols
)

TARGET = 'sales'

print(f'Features categoricas ({len(CAT_FEATURES)}): {CAT_FEATURES}')
print(f'Features numericas ({len(NUM_FEATURES)}): {NUM_FEATURES}')
print(f'Total features: {len(CAT_FEATURES) + len(NUM_FEATURES)}')

Features categoricas (7): ['category', 'department', 'store_code', 'region', 'season', 'pay_period', 'weekday']
Features numericas (22): ['sell_price', 'is_holiday', 'is_weekend', 'dayofweek', 'dayofmonth', 'weekofyear', 'month', 'year', 'week', 'indice_estacional_store_item', 'lag_28', 'lag_35', 'lag_42', 'rolling_mean_7', 'rolling_std_7', 'rolling_median_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_median_14', 'rolling_mean_28', 'rolling_std_28', 'rolling_median_28']
Total features: 29


In [10]:
print(df_test[['date', 'id', 'sales', 'lag_28', 'rolling_mean_7']].head(20))
print(f'\nNaN en lags del test:')
print(df_test[lag_cols].isna().sum())

               date                      id  sales  lag_28  rolling_mean_7
13637806 2016-03-28  ACCESORIES_1_001_BOS_1      1     0.0        0.285714
13646953 2016-03-29  ACCESORIES_1_001_BOS_1      0     1.0        0.428571
13656100 2016-03-30  ACCESORIES_1_001_BOS_1      1     0.0        0.285714
13665247 2016-03-31  ACCESORIES_1_001_BOS_1      0     1.0        0.428571
13674394 2016-04-01  ACCESORIES_1_001_BOS_1      0     0.0        0.285714
13683541 2016-04-02  ACCESORIES_1_001_BOS_1      0     1.0        0.428571
13692688 2016-04-03  ACCESORIES_1_001_BOS_1      0     0.0        0.428571
13701835 2016-04-04  ACCESORIES_1_001_BOS_1      1     0.0        0.428571
13710982 2016-04-05  ACCESORIES_1_001_BOS_1      0     1.0        0.428571
13720129 2016-04-06  ACCESORIES_1_001_BOS_1      1     0.0        0.428571
13729276 2016-04-07  ACCESORIES_1_001_BOS_1      0     1.0        0.428571
13738423 2016-04-08  ACCESORIES_1_001_BOS_1      0     0.0        0.428571
13747570 2016-04-09  ACCE

---
## 4. Modelo 1 - XGBoost

XGBoost es un modelo de gradient boosting sobre árboles de decisión. No es un modelo
de series temporales nativo, pero con los lag features y rolling features construidos
arriba puede aprender los patrones.

**Ventajas para este problema:**
- Un solo modelo para todas las series (no hay que entrenar uno por producto x tienda)
- Maneja bien variables categóricas y numéricas mezcladas
- Escala bien con millones de filas
- Interpretable via feature importance

In [11]:
# XGBoost puede manejar categorías directamente con enable_categorical=True
for col in CAT_FEATURES:
    df_train[col] = df_train[col].astype('category')
    df_test[col]  = df_test[col].astype('category')

ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

X_train = df_train[ALL_FEATURES]
y_train = df_train[TARGET]
X_test  = df_test[ALL_FEATURES]
y_test  = df_test[TARGET]

In [12]:
# Parámetros iniciales razonables — se optimizarán con tuning posterior
xgb_model = xgb.XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 10,   # regularización: evita overfitting en series con pocas ventas
    tree_method       = 'hist',
    enable_categorical= True,
    random_state      = RANDOM_SEED,
    n_jobs            = -1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

print('Entrenamiento completadooooo')

[0]	validation_0-rmse:2.96848
[100]	validation_0-rmse:1.87895
[200]	validation_0-rmse:1.87403
[300]	validation_0-rmse:1.87294
[400]	validation_0-rmse:1.87245
[499]	validation_0-rmse:1.87204
Entrenamiento completadooooo


In [13]:
y_pred_xgb = xgb_model.predict(X_test)
y_pred_xgb = np.clip(y_pred_xgb, 0, None)  # las ventas no pueden ser negativas

mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print(f'XGBoost - MAE:  {mae_xgb:.4f}')
print(f'XGBoost - RMSE: {rmse_xgb:.4f}')

XGBoost - MAE:  0.9817
XGBoost - RMSE: 1.8720


In [14]:
# Feature importance: qué variables explican más las predicciones
importance = pd.DataFrame({
    'feature'   : ALL_FEATURES,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

fig = px.bar(importance.head(20), x='importance', y='feature', orientation='h',
             title='Top 20 features más importantes - XGBoost',
             labels={'importance': 'Importancia', 'feature': 'Feature'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [15]:
# Visualización: predicción vs real en una serie de ejemplo
#serie_ejemplo = df_test['id'].value_counts().index[0]
# En vez de coger el primero, busca uno con más ventas en el test
serie_ejemplo = df_test.groupby('id')['sales'].mean().idxmax()

mask  = df_test['id'] == serie_ejemplo
fechas = df_test.loc[mask, 'date']
real   = y_test[mask]
pred   = y_pred_xgb[mask]

fig = go.Figure()
fig.add_trace(go.Scatter(x=fechas, y=real, name='Real',       line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=fechas, y=pred, name='Predicción', line=dict(color='tomato', dash='dash')))
fig.update_layout(title=f'Predicción vs Real - {serie_ejemplo}',
                  xaxis_title='Fecha', yaxis_title='Ventas')
fig.show()

---
## 5. Modelo 2 - Prophet

Prophet es un modelo desarrollado por Meta para series temporales con estacionalidad
múltiple y efectos de festivos. A diferencia de XGBoost, modela cada serie de forma independiente.

**Ventajas para este problema:**
- Maneja estacionalidades múltiples (semanal, anual) de forma automática
- Permite añadir festivos y eventos como regressors externos
- Produce intervalos de confianza en las predicciones

**Limitación:** con ~30.000 series (3.049 productos x 10 tiendas) hay que decidir
si se entrena un modelo por serie o se selecciona un subconjunto representativo.




In [21]:
from prophet import Prophet

serie_id = df['id'].value_counts().index[0]
serie_df = df[df['id'] == serie_id][['date', 'sales']].rename(columns={'date': 'ds', 'sales': 'y'})

train_prophet = serie_df[serie_df['ds'] <= fecha_corte]
test_prophet  = serie_df[serie_df['ds'] >  fecha_corte]

prophet_model = Prophet(
    yearly_seasonality = True,
    weekly_seasonality = True,
    daily_seasonality  = False,
    seasonality_mode   = 'multiplicative'
)
prophet_model.fit(train_prophet)

future   = prophet_model.make_future_dataframe(periods=28)
forecast = prophet_model.predict(future)

mae_prophet = mean_absolute_error(test_prophet['y'], forecast.tail(28)['yhat'].clip(0))
print(f'Prophet - MAE: {mae_prophet:.4f}')
rmse_prophet = np.sqrt(mean_squared_error(test_prophet['y'], forecast.tail(28)['yhat'].clip(0)))
print(f'Prophet - RMSE: {rmse_prophet:.4f}')


22:37:52 - cmdstanpy - INFO - Chain [1] start processing
22:37:52 - cmdstanpy - INFO - Chain [1] done processing


Prophet - MAE: 1.0162
Prophet - RMSE: 1.2577


---
## 6. Modelo 3 - SARIMAX

SARIMAX (Seasonal ARIMA with eXogenous variables) es un modelo estadístico clásico
para series temporales. Modela la autocorrelación de la serie y permite añadir
variables externas (precio, festivos) como regressors.

**Ventajas para este problema:**
- Muy interpretable: los parámetros tienen significado estadístico directo
- Robusto para series con patrones estacionales bien definidos
- Produce intervalos de confianza

**Limitación:** computacionalmente costoso a escala. Se aplicará sobre series
representativas por cluster.

> **Estado:** pendiente de implementación completa.

In [22]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

serie_id  = df['id'].value_counts().index[0]
serie_df  = df[df['id'] == serie_id].set_index('date').sort_index()

train_sarimax = serie_df[serie_df.index <= fecha_corte]
test_sarimax  = serie_df[serie_df.index >  fecha_corte]

exog_train = train_sarimax[['sell_price', 'is_holiday']]
exog_test  = test_sarimax[['sell_price', 'is_holiday']]

# Orden (p,d,q)(P,D,Q,s) — s=7 para estacionalidad semanal
# Los parámetros óptimos se determinarán con auto_arima (pmdarima)
sarimax_model = SARIMAX(
    train_sarimax['sales'],
    exog           = exog_train,
    order          = (1, 1, 1),
    seasonal_order = (1, 1, 1, 7)
).fit(disp=False)

pred_sarimax = sarimax_model.forecast(steps=28, exog=exog_test)
mae_sarimax  = mean_absolute_error(test_sarimax['sales'], pred_sarimax.clip(0))
print(f'SARIMAX - MAE: {mae_sarimax:.4f}')
rmse_sarimax = np.sqrt(mean_squared_error(test_sarimax['sales'], pred_sarimax.clip(0)))
print(f'SARIMAX - RMSE: {rmse_sarimax:.4f}')



C:\Users\71726970\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency D will be used.

C:\Users\71726970\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency D will be used.



SARIMAX - MAE: 0.9047
SARIMAX - RMSE: 1.2053


---
## 7. Comparativa de modelos y selección

Una vez implementados los tres modelos, compararemos su rendimiento sobre el mismo
conjunto de test usando las siguientes métricas:

- **MAE** (Mean Absolute Error): error medio en unidades. Fácil de interpretar para negocio.
- **RMSE** (Root Mean Squared Error): penaliza más los errores grandes.

In [23]:
# Métricas utilizadas:
# MAE  - error medio en unidades, fácil de interpretar para negocio
# RMSE - penaliza más los errores grandes, útil para detectar series problemáticas
resultados = pd.DataFrame({
    'Modelo' : ['XGBoost', 'Prophet', 'SARIMAX'],
    'MAE'    : [mae_xgb,    mae_prophet,  mae_sarimax],
    'RMSE'   : [rmse_xgb,   rmse_prophet, rmse_sarimax],
})
print(resultados.to_string(index=False))

 Modelo      MAE     RMSE
XGBoost 0.981735 1.872036
Prophet 1.016169 1.257741
SARIMAX 0.904659 1.205255


---
## 8. Generación de predicciones finales

Según enunciado: predicciones en formato producto x tienda para los 28 dias
del conjunto de test que se proporcionará al final del proyecto.

Una fila por serie (`id` = item + store_code), una columna por dia de predicción (F1..F28).

In [ ]:
# submission = pd.DataFrame({'id': df_test['id'].unique()})
# for i, fecha in enumerate(sorted(df_test['date'].unique()), 1):
#     mask = df_test['date'] == fecha
#     submission[f'F{i}'] = y_pred_xgb[mask.values]

# submission.to_csv(DATA_PATH + 'submission_xgb.csv', index=False)
# print(f'Submission guardada: {submission.shape}')
# submission.head()

print('Pendiente: se generará con el modelo final seleccionado en la sección 7.')